# 单 repeat 证据封装入口

该 Notebook 只把一个已登记 seed-key repeat 的7类活动随机化 leaf ZIP 按原始字节封装为自包含证据组件, 并复制到 Google Drive 的 `randomization_repeat_evidence/<repeat>` 目录。它不重建攻击矩阵、共同协议、论文表图或完整结果包。

运行依赖: 当前 repeat 的主方法、正式消融、dataset-level 质量、3个 method-faithful baseline 与 T2SMark 结果包均已写入 Drive。3类跨 repeat 不变 official-reference 包不进入该组件, 由最终聚合层只选择一次。该入口不需要 GPU。

Notebook 只调用统一宿主入口, 不承载证据选择、校验或打包实现。


## 运行前准备

1. 确认 `SLM_WM_RANDOMIZATION_REPEAT_ID` 与前序 GPU Notebook 使用的 repeat 完全一致。
2. 确认当前 repeat 的7类活动随机化结果包已经写入当前论文层级的 Drive 结果根目录。
3. 该入口不需要 GPU, 但需要访问 Google Drive 或已同步的本地结果包目录。
4. 输出写入 `randomization_repeat_evidence/<repeat>`; 单 repeat 组件固定不支持论文结论。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"
SLM_WM_RANDOMIZATION_REPEAT_ID = "seed_00_key_00"

import os

from google.colab import drive

drive.mount('/content/drive')
# 可切换为 "probe_paper"、"pilot_paper" 或 "full_paper"。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME
os.environ["SLM_WM_RANDOMIZATION_REPEAT_ID"] = SLM_WM_RANDOMIZATION_REPEAT_ID


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
print({"workspace_dir": str(workspace_dir), "repository_commit": repository_commit})


In [ ]:
FORMAL_HOST_LAUNCHER = "scripts/run_formal_workflow_host.py"
print({"formal_host_launcher": FORMAL_HOST_LAUNCHER})


In [ ]:
# 论文运行配置由精确 workflow_orchestrator 子解释器统一加载.


In [ ]:
from pathlib import Path

package_search_root = Path("/content/drive/MyDrive/SLM") / f"{SLM_WM_PAPER_RUN_NAME}_results"
assert package_search_root.exists(), f"未找到当前论文运行层级的 Google Drive 结果目录: {package_search_root}"
zip_count_by_dir = {
    child.name: len(list(child.glob("*.zip")))
    for child in sorted(package_search_root.iterdir())
    if child.is_dir()
}
print({"package_search_root": str(package_search_root), "zip_count_by_dir": zip_count_by_dir})


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

drive_result_root = Path("/content/drive/MyDrive/SLM") / f"{SLM_WM_PAPER_RUN_NAME}_results"
package_search_root = drive_result_root
result_path = Path("outputs/formal_workflow_execution") / SLM_WM_PAPER_RUN_NAME / "randomization_repeat_evidence" / "workflow_result.json"
host_command = [
    sys.executable,
    "-I",
    FORMAL_HOST_LAUNCHER,
    "--root",
    ".",
    "--repository-commit",
    repository_commit,
    "repeat_evidence",
    "--paper-run-name",
    SLM_WM_PAPER_RUN_NAME,
    "--package-search-root",
    package_search_root.as_posix(),
    "--randomization-repeat-id",
    SLM_WM_RANDOMIZATION_REPEAT_ID,
    "--result-path",
    result_path.as_posix(),
]
subprocess.run(host_command, check=True)
formal_workflow_result = json.loads(result_path.read_text(encoding="utf-8"))
assert formal_workflow_result["decision"] == "pass", formal_workflow_result
archive_path = Path(formal_workflow_result["archive_record"]["archive_path"])
repeat_id = formal_workflow_result["archive_record"]["randomization_repeat_id"]
drive_archive_dir = drive_result_root / "randomization_repeat_evidence" / repeat_id
drive_archive_dir.mkdir(parents=True, exist_ok=True)
drive_archive_path = drive_archive_dir / archive_path.name
shutil.copy2(archive_path, drive_archive_path)
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == hashlib.sha256(drive_archive_path.read_bytes()).hexdigest()
formal_workflow_result["drive_archive_path"] = drive_archive_path.as_posix()
formal_workflow_result


In [ ]:
print(json.dumps(formal_workflow_result, ensure_ascii=False, sort_keys=True, indent=2))
